In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval
except Exception:
    pass

In [2]:
import numpy as np
from numpy.typing import NDArray

from dataeval.config import set_seed
from dataeval.core import image_stats
from dataeval.data import Filter, View
from dataeval.flags import ImageStats
from dataeval.protocols import DatasetMetadata, DatumMetadata
from dataeval.quality import Outliers

set_seed(0)  # For reproducibility

In [3]:
HEIGHT, WIDTH = 64, 96
rng = np.random.default_rng(0)


class Frames:
    """Constructed frames stand in for yours: 40 from a clear sensor and 10 from a sensor with a painted hood."""

    def __init__(self) -> None:
        self.metadata = DatasetMetadata(id="frames", index2label={0: "road", 1: "field"})
        self.sensors = ["hood" if i % 5 == 4 else "clear" for i in range(50)]
        self.images: list[NDArray[np.uint8]] = []
        for sensor in self.sensors:
            scene = rng.integers(60, 180, (HEIGHT // 4, WIDTH // 4)).repeat(4, 0).repeat(4, 1)
            frame = np.clip(scene + rng.normal(0, 6, (HEIGHT, WIDTH)), 0, 255).astype(np.uint8)
            if sensor == "hood":
                frame[-20:] = 235
            self.images.append(np.stack([frame] * 3))

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, index: int) -> tuple[NDArray[np.uint8], NDArray[np.float32], DatumMetadata]:
        target = np.eye(2, dtype=np.float32)[index % 2]  # Filter needs targets, and the guide does not use them
        return self.images[index], target, DatumMetadata(id=index, **{"sensor": self.sensors[index]})


dataset = Frames()

In [4]:
hood_mask = np.zeros((HEIGHT, WIDTH), dtype=bool)
hood_mask[-20:] = True

masks = {"clear": None, "hood": hood_mask}
splits = {sensor: View(dataset, Filter(where=f"sensor = '{sensor}'")) for sensor in masks}
print({sensor: len(split) for sensor, split in splits.items()})

{'clear': 40, 'hood': 10}


In [5]:
stats = ImageStats.DIMENSION | ImageStats.PIXEL | ImageStats.VISUAL
results = [image_stats(split, stats=stats, mask=masks[sensor]) for sensor, split in splits.items()]

print("share of a hood frame removed:", results[1].get("details", {})["masked_fraction"][0])

share of a hood frame removed: 0.3125


In [6]:
def flagged(measured):
    """Return the dataset indices of the frames that Outliers flags, given the results of the splits in order."""
    found = Outliers().from_stats(measured)
    return sorted(
        index
        for n, split in enumerate(splits.values())
        for index in split.resolve_indices(found.filter(dataset_index=n)["item_index"].unique().to_list())
    )


unmasked = [image_stats(split, stats=stats) for split in splits.values()]
without_mask, with_mask = flagged(unmasked), flagged(results)
print("flagged without the mask:", without_mask)
print("flagged with the mask:", with_mask)

flagged without the mask: [4, 9, 14, 19, 24, 29, 34, 39, 44, 49]
flagged with the mask: []


In [7]:
# TEST ASSERTION CELL ###
hood_frames = {i for i, sensor in enumerate(dataset.sensors) if sensor == "hood"}
assert len(dataset) == 50
assert len(hood_frames) == 10
assert {sensor: len(split) for sensor, split in splits.items()} == {"clear": 40, "hood": 10}
assert results[1].get("details", {})["masked_fraction"][0] == 20 / 64
assert hood_frames <= set(without_mask)
assert not hood_frames & set(with_mask)